# 3.1 DNN Regression

使用 Diabetes Dataset 建立表格資料 DNN 迴歸模型。這份 notebook 示範資料切分、標準化、Dense DNN 建模、EarlyStopping、MAE/RMSE/R2 評估、逐筆預測與模型保存。

## 1. 載入套件

TensorFlow/Keras 負責建立與訓練模型；scikit-learn 負責載入資料、切分資料、標準化與計算迴歸指標。

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

tf.keras.utils.set_random_seed(42)

## 2. 載入 Diabetes Dataset

Diabetes Dataset 是 scikit-learn 內建的小型迴歸資料集。每筆資料包含 10 個數值特徵，目標值是疾病進展的連續數值，適合示範表格 DNN regression。

In [ ]:
data = load_diabetes(as_frame=True)
df = data.frame.copy()
print(df.shape)
df.head()

## 3. 切分資料與標準化

這裡先切出 test set，再從訓練資料中切出 validation set。Train 用來更新模型權重，validation 用來監控訓練過程，test 保留到最後評估泛化能力。

標準化只能在訓練集上 `fit`。Validation/test 只能使用訓練集學到的平均值與標準差做 `transform()`，避免資料洩漏。

In [ ]:
X = data.data.values.astype('float32')
y = data.target.values.astype('float32')

x_train_full, x_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
x_train, x_valid, y_train, y_valid = train_test_split(
    x_train_full, y_train_full, test_size=0.2, random_state=42
)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train).astype('float32')
x_valid = scaler.transform(x_valid).astype('float32')
x_test = scaler.transform(x_test).astype('float32')

print('x_train:', x_train.shape)
print('x_valid:', x_valid.shape)
print('x_test:', x_test.shape)

## 4. 建立 DNN 迴歸模型

迴歸輸出層使用 1 個線性神經元，不使用 sigmoid 或 softmax。Loss 使用 MSE，監控指標使用 MAE。MSE 對大誤差較敏感，MAE 的單位則和目標值相同，較容易解讀。

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(x_train.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1)
])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01), loss='mse', metrics=['mae'])
model.summary()

## 5. 訓練模型

`EarlyStopping` 監控 validation loss。當 validation loss 長時間沒有改善時停止訓練，並透過 `restore_best_weights=True` 回復到 validation loss 最好的模型權重。

In [ ]:
callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)]
history = model.fit(
    x_train, y_train,
    validation_data=(x_valid, y_valid),
    epochs=300,
    batch_size=32,
    callbacks=callbacks,
    verbose=0
)
print('實際訓練 epochs:', len(history.history['loss']))

## 6. 評估模型

迴歸任務除了 loss，也常看 MAE、RMSE 與 R2。不要只看 train 指標；valid/test 才能反映模型是否能泛化到未見資料。

In [ ]:
def regression_report(model, x, y, name):
    pred = model.predict(x, verbose=0).ravel()
    return {
        'dataset': name,
        'mae': mean_absolute_error(y, pred),
        'rmse': np.sqrt(mean_squared_error(y, pred)),
        'r2': r2_score(y, pred),
    }

report = pd.DataFrame([
    regression_report(model, x_train, y_train, 'train'),
    regression_report(model, x_valid, y_valid, 'valid'),
    regression_report(model, x_test, y_test, 'test'),
])
report

## 7. 預測新資料

模型預測結果是連續數值。將實際值、預測值與誤差放在同一張表中，可以觀察模型在哪些樣本上偏高或偏低。

In [ ]:
sample = x_test[:5]
pred = model.predict(sample, verbose=0).ravel()
pd.DataFrame({'actual': y_test[:5], 'predicted': pred, 'error': pred - y_test[:5]}).round(2)

## 8. 觀察訓練曲線

訓練曲線可以用來判斷模型是否收斂，以及 validation loss 是否開始惡化。若 train loss 持續下降但 valid loss 上升，通常代表過擬合。

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='valid loss')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('DNN Regression Training Curve')
plt.legend()
plt.show()

## 9. 保存與載入模型

這裡使用 `.keras` 格式保存模型，再重新載入並確認載入前後的預測值一致。若標準化流程在模型外進行，正式部署時也要保存同一個 scaler。

In [ ]:
MODEL_DIR = Path('outputs/models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODEL_DIR / 'dnn_regression_model.keras'
model.save(model_path)

loaded_model = tf.keras.models.load_model(model_path)
loaded_pred = loaded_model.predict(sample, verbose=0).ravel()
pd.DataFrame({'pred_before_save': pred, 'pred_after_load': loaded_pred}).round(4)

## 10. 小結

DNN Regression 的基本流程是：準備數值特徵、切分 train/validation/test、標準化資料、建立 Dense DNN、用 MSE/MAE 訓練與評估，最後保存模型供推論使用。